# Классификация местоположения по спутниковым изображениям и аудио

В этом ноутбуке строятся **две отдельные модели** для 13 классов местоположений:

- **ViT (Vision Transformer)** — получает спутниковое изображение и предсказывает класс.
- **AST (Audio Spectrogram Transformer)** — получает аудиозапись `.ogg` и предсказывает класс.

После обучения ноутбук:

1. оценивает модели на внутренней размеченной выборке, выделенной из `train`;
2. сравнивает Accuracy и Macro F1, а также показывает ошибки по каждому классу;
3. при наличии надёжных пар JPG/OGG проверяет, помогает ли blending;
4. выполняет **пакетный инференс по всем файлам** в `image/val` и `audio/val` с помощью `Trainer.predict()`;
5. сохраняет отдельные CSV и общий файл `M1_{lastname}_{firstname}_submission.csv`.

> Важно: в папках `val` правильных меток нет. Поэтому их нельзя использовать для вычисления Accuracy или F1. Они предназначены для получения итоговых предсказаний.

## Ожидаемая структура папок

```text
dataset/
├── image/
│   ├── train/
│   │   ├── class_1/       # внутри JPG/PNG одного класса
│   │   ├── class_2/
│   │   └── ...            # всего 13 папок классов
│   └── val/               # плоская папка без подпапок классов
│       ├── 001.jpg
│       ├── 002.jpg
│       └── ...
└── audio/
    ├── train/
    │   ├── class_1/       # внутри OGG одного класса
    │   ├── class_2/
    │   └── ...            # те же 13 классов
    └── val/               # плоская папка без подпапок классов
        ├── 001.ogg
        ├── 002.ogg
        └── ...
```

Названия папок в `image/train` и `audio/train` должны описывать одни и те же классы. Количество JPG и OGG **не обязано совпадать**.

### Как читать ноутбук

Запускай ячейки сверху вниз. Перед обучением проверь путь к данным и настройки. Для первой проверки можно уменьшить `NUM_EPOCHS`; после тестового запуска верни полноценное значение.

## Как читать объяснения в этом ноутбуке

Перед каждой важной кодовой ячейкой есть отдельный блок **«Разбор кода»**. В нём объясняется, зачем нужен каждый используемый метод или функция, какие аргументы им передаются, что они возвращают и где результат используется дальше. Внутри кода также оставлены короткие комментарии в местах, где легко потерять логику.

При изучении двигайся сверху вниз: сначала прочитай объяснение, затем запусти код и проверь вывод. Особое внимание уделяй структуре возвращаемых объектов (`Dataset`, словари, тензоры и массивы logits), потому что понимание формата данных помогает самостоятельно переносить готовые примеры на новые задачи.


## 0. Установка библиотек

Используем `datasets` для загрузки `ImageFolder` и `AudioFolder`, `transformers` для ViT/AST и `Trainer`, а `torchvision` — для преобразований изображений.

Для чтения `.ogg` устанавливаем FFmpeg и TorchCodec. В Google Colab после установки пакетов иногда требуется перезапустить среду выполнения. Если Colab предложит перезапуск, сделай его и продолжи с первой ячейки.

### Разбор команды установки

Это команда установки библиотек прямо в окружение текущего Jupyter/Colab:

- `%pip` — специальная команда IPython. В отличие от обычного `pip` в терминале, она устанавливает пакеты в Python-окружение, которое использует текущий ноутбук.
- `-q` (*quiet*) — уменьшает количество служебного вывода.
- `-U` (*upgrade*) — обновляет пакет, если он уже установлен.
- `datasets` — чтение и хранение наборов данных `Dataset` / `DatasetDict`, в том числе изображений и аудио.
- `transformers` — предварительно обученные ViT и AST, обработчики данных и `Trainer`.
- `accelerate` — помогает `Trainer` работать с CPU/GPU и управлять вычислениями.
- `scikit-learn` — метрики классификации и матрицы ошибок.
- `pandas` — таблицы результатов и CSV.
- `matplotlib` — графики распределения классов и матрицы ошибок.
- `torchvision` — преобразования изображений: обрезка, изменение размера, перевод в тензор и нормализация.
- `torchcodec` и `soundfile` — декодирование и чтение аудиофайлов; `torchcodec` используется современными версиями Datasets для аудио.

Версии `datasets` и `transformers` заданы как нижние границы (`>=`), а не как точные версии. Поэтому при повторном запуске в будущем могут установиться более новые версии. Для строго воспроизводимого окружения обычно фиксируют точные версии.

In [ ]:
%pip -q install -U "datasets>=4.4.0" "transformers>=4.46.0" accelerate scikit-learn pandas matplotlib torchvision torchcodec soundfile

### Разбор команды FFmpeg

`apt-get` — системный менеджер пакетов Ubuntu, а не установщик Python-библиотек.

- `apt-get -qq update` обновляет локальный список доступных системных пакетов; `-qq` делает вывод очень коротким.
- `&&` запускает следующую команду только в том случае, если первая завершилась успешно.
- `apt-get -qq install -y ffmpeg` устанавливает FFmpeg; флаг `-y` автоматически подтверждает установку.

FFmpeg нужен для декодирования некоторых аудиоформатов, в том числе OGG. Он не обучает модель и не преобразует метки классов — это системный инструмент чтения медиафайлов.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg

## 1. Подключение Google Drive

Эта ячейка нужна для Google Colab. Если запускаешь ноутбук локально, пропусти её и укажи свой путь в следующем разделе.

### Разбор `drive.mount()`

`from google.colab import drive` импортирует модуль подключения Google Drive, доступный в Google Colab.

`drive.mount("/content/drive")` подключает твой Google Drive к файловой системе виртуальной машины Colab. После успешного подключения файлы становятся доступны как обычные пути, например `/content/drive/MyDrive/dataset`.

Аргумент `"/content/drive"` — точка монтирования, то есть каталог, через который ноутбук будет обращаться к диску. Если работаешь не в Colab, эту ячейку пропускают, а `DATA_ROOT` ниже указывают на локальный каталог.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Импорты и настройки проекта

В этой секции задаются пути и параметры. Для видеокарты с небольшой памятью начальные размеры батчей специально выбраны небольшими: ViT — 4 изображения, AST — 1 аудиозапись.

**`gradient_accumulation_steps`** позволяет накапливать градиенты нескольких маленьких батчей и реже обновлять веса. Это не означает, что все эти примеры одновременно находятся в видеопамяти.

### Разбор импортов и проверки окружения

Импорт — это подключение готовых классов и функций, чтобы использовать их в ноутбуке.

**Работа с файлами и памятью:**
- `Path` — удобное построение путей к папкам и файлам; оператор `/` объединяет части пути.
- `Counter` — считает, сколько раз встречается каждый идентификатор класса.
- `defaultdict(list)` — словарь, который автоматически создаёт пустой список для нового ключа; ниже он группирует файлы с одинаковыми идентификаторами.
- `gcd` — наибольший общий делитель; нужен для вычисления коэффициентов пересчёта частоты дискретизации.
- `gc` — сборщик мусора Python; `gc.collect()` просит освободить объекты, на которые больше нет ссылок.
- `warnings` — вывод предупреждений, которые не обязательно останавливают выполнение.

**Числа и таблицы:** `numpy` (`np`) нужен для массивов и операций над logits; `pandas` (`pd`) — для таблиц; `matplotlib.pyplot` (`plt`) — для графиков.

**PyTorch:** `torch` — тензоры, GPU и вычисления; `torch.nn.functional` (`F`) — в частности, функция потерь `cross_entropy`.

**Аудио:** `resample_poly` меняет частоту дискретизации сигнала; `softmax` переводит logits в вероятности, сумма которых по классам равна 1.

**Метрики scikit-learn:** `accuracy_score`, `f1_score`, `classification_report`, `confusion_matrix` и `ConfusionMatrixDisplay` вычисляют метрики и показывают ошибки классификации.

**Hugging Face Datasets:** `load_dataset` загружает папки в Dataset; `DatasetDict` хранит несколько частей; `HFImage` и `Audio` описывают столбцы изображений/аудио; `ClassLabel` задаёт именованные классы.

**Обработка изображений:** `Compose` объединяет преобразования в последовательность; `RandomResizedCrop` делает случайную обрезку; `Resize` меняет размер; `CenterCrop` берёт центральную часть; `ToTensor` превращает PIL-изображение в тензор; `Normalize` нормализует каналы.

**Transformers:**
- `set_seed` фиксирует случайные генераторы, насколько это возможно, для воспроизводимости.
- `AutoImageProcessor` загружает настройки обработки изображений конкретной модели.
- `AutoFeatureExtractor` загружает обработчик аудио для AST.
- `AutoModelForImageClassification` и `AutoModelForAudioClassification` загружают модели соответствующих задач.
- `DefaultDataCollator` собирает отдельные примеры в батч.
- `TrainingArguments` хранит настройки обучения.
- `Trainer` организует обучение, оценку и предсказания.

`set_seed(42)` устанавливает фиксированное начальное значение генераторов случайных чисел. `42` — произвольное выбранное число, а не специальный параметр модели.

`torch.cuda.is_available()` возвращает `True`, если PyTorch видит CUDA-совместимую видеокарту. `torch.cuda.get_device_name(0)` возвращает имя первого GPU. `torch.__version__` показывает установленную версию PyTorch.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from math import gcd
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from scipy.signal import resample_poly
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from datasets import (
    load_dataset,
    DatasetDict,
    Image as HFImage,
    Audio,
    ClassLabel,
)

from torchvision.transforms import (
    Compose,
    RandomResizedCrop,
    Resize,
    CenterCrop,
    ToTensor,
    Normalize,
)

from transformers import (
    set_seed,
    AutoImageProcessor,
    AutoFeatureExtractor,
    AutoModelForImageClassification,
    AutoModelForAudioClassification,
    DefaultDataCollator,
    TrainingArguments,
    Trainer,
)

set_seed(42)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Настройки проекта

В `DATA_ROOT` укажи родительскую папку, внутри которой расположены `image/` и `audio/`.

Если памяти GPU не хватает:

- уменьши `IMAGE_BATCH_SIZE` до 2 или 1;
- для AST оставь `AUDIO_BATCH_SIZE = 1`;
- при необходимости уменьши `NUM_EPOCHS` для пробного запуска.

`USE_CLASS_WEIGHTS=False` оставляем на первом эксперименте. Сначала важно получить базовые результаты, а затем решать, помогает ли взвешивание редких классов.

### Разбор настройки путей и параметров

`Path("...")` создаёт объект пути. Выражение `DATA_ROOT / "image" / "train"` объединяет части пути безопаснее, чем вручную собирать строки через `+` или обратные слеши.

- `DATA_ROOT` — корневая папка проекта. Внутри неё должны быть каталоги `image/` и `audio/`.
- `IMAGE_TRAIN_DIR`, `IMAGE_VAL_DIR`, `AUDIO_TRAIN_DIR`, `AUDIO_VAL_DIR` — пути к четырём исходным наборам.
- `OUTPUT_DIR` — место сохранения весов моделей и CSV.
- `OUTPUT_DIR.mkdir(parents=True, exist_ok=True)` создаёт папку результатов. `parents=True` создаёт недостающие родительские каталоги; `exist_ok=True` не выдаёт ошибку, если папка уже существует.
- `LASTNAME` и `FIRSTNAME` подставляются в имя итогового submission-файла.
- `SEED` — число для воспроизводимости случайного разбиения и инициализации.
- `NUM_EPOCHS` — количество полных проходов по обучающей выборке.
- `*_BATCH_SIZE` — сколько примеров обрабатывается за один шаг на одном устройстве. Для AST размер меньше, так как спектральные тензоры могут занимать много памяти.
- `*_GRAD_ACCUM` — сколько небольших батчей накапливают градиенты перед обновлением весов. Например, batch 4 и accumulation 4 дают эффективный batch около 16, но не требуют держать все 16 изображений одновременно в памяти.
- `USE_CLASS_WEIGHTS` включает или отключает взвешенную функцию потерь для редких классов.
- `CLASS_WEIGHT_POWER` регулирует силу компенсации дисбаланса.
- `MIN_PAIRED_PER_CLASS_FOR_BLEND` — минимальное число совпадающих JPG/OGG на класс, необходимое, чтобы пытаться строить общую validation для blending.

Цикл `for folder in [...]` проверяет наличие каждой обязательной папки. `folder.exists()` возвращает `True`, если путь существует. `raise FileNotFoundError(...)` останавливает ноутбук с понятным сообщением, если путь задан неправильно. Лучше обнаружить ошибку до долгой загрузки и обучения.

In [ ]:
DATA_ROOT = Path("/content/drive/MyDrive/dataset")

IMAGE_TRAIN_DIR = DATA_ROOT / "image" / "train"
IMAGE_VAL_DIR   = DATA_ROOT / "image" / "val"
AUDIO_TRAIN_DIR = DATA_ROOT / "audio" / "train"
AUDIO_VAL_DIR   = DATA_ROOT / "audio" / "val"

OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Замени на свои фамилию и имя перед созданием итогового CSV.
LASTNAME = "Lastname"
FIRSTNAME = "Firstname"

SEED = 42
NUM_EPOCHS = 5

IMAGE_BATCH_SIZE = 4
IMAGE_EVAL_BATCH_SIZE = 8
IMAGE_GRAD_ACCUM = 4

AUDIO_BATCH_SIZE = 1
AUDIO_EVAL_BATCH_SIZE = 1
AUDIO_GRAD_ACCUM = 8

# Сначала запускаем базовый вариант без балансировки.
USE_CLASS_WEIGHTS = False
CLASS_WEIGHT_POWER = 0.5

# Блендинг будет подбираться только если есть достаточное число пар в train.
MIN_PAIRED_PER_CLASS_FOR_BLEND = 10

for folder in [IMAGE_TRAIN_DIR, IMAGE_VAL_DIR, AUDIO_TRAIN_DIR, AUDIO_VAL_DIR]:
    if not folder.exists():
        raise FileNotFoundError(
            f"Не найдена папка: {folder}\n"
            "Проверь DATA_ROOT и ожидаемую структуру каталогов."
        )

print("Все четыре папки найдены.")
print("Результаты будут сохранены в:", OUTPUT_DIR)

## 4. Загрузка четырёх наборов данных через Hugging Face Datasets

`ImageFolder` и `AudioFolder` автоматически создают `label` по названию подпапки. Поэтому для `train` каждый файл получает правильный числовой класс без ручного создания таблицы меток.

Папки `val` плоские: внутри лежат только файлы, а не папки классов. Указываем `drop_labels=True`, чтобы библиотека не создавала метки.

В результате появятся четыре набора:

- `image_train_all`: столбцы `image`, `label`;
- `audio_train_all`: столбцы `audio`, `label`;
- `image_val`: только `image`;
- `audio_val`: только `audio`.

### Разбор `load_dataset()` для четырёх папок

`load_dataset()` — функция Hugging Face Datasets, которая читает файлы и формирует объект `Dataset`.

Общие аргументы:
- первый аргумент (`"imagefolder"` или `"audiofolder"`) выбирает загрузчик и тип файлов;
- `data_dir=...` — каталог, который надо прочитать;
- `split="train"` просит вернуть одну часть набора, которую сам загрузчик в данном случае назовёт `train`;
- `drop_labels=True` отключает создание меток, когда в папке `val` нет подпапок с правильными классами.

Что получается:
- `image_train_all` содержит `image` и `label`;
- `audio_train_all` содержит `audio` и `label`;
- `image_val` содержит только `image`;
- `audio_val` содержит только `audio`.

В `train` названия подпапок становятся классами, а каждому классу назначается числовой ID. В плоской папке `val` правильных ответов нет, поэтому мы намеренно не создаём столбец `label`.

`print()` выводит краткое представление Dataset: количество строк и названия/типы столбцов. Это первая проверка того, что данные прочитались ожидаемым образом.

In [ ]:
image_train_all = load_dataset(
    "imagefolder",
    data_dir=str(IMAGE_TRAIN_DIR),
    split="train",
)

audio_train_all = load_dataset(
    "audiofolder",
    data_dir=str(AUDIO_TRAIN_DIR),
    split="train",
)

image_val = load_dataset(
    "imagefolder",
    data_dir=str(IMAGE_VAL_DIR),
    split="train",
    drop_labels=True,
)

audio_val = load_dataset(
    "audiofolder",
    data_dir=str(AUDIO_VAL_DIR),
    split="train",
    drop_labels=True,
)

print("IMAGE TRAIN:")
print(image_train_all)
print("\nAUDIO TRAIN:")
print(audio_train_all)
print("\nIMAGE VAL:")
print(image_val)
print("\nAUDIO VAL:")
print(audio_val)

## 5. Проверка классов и соответствия меток

У изображений и аудио должны быть одинаковые названия 13 классов. Порядок папок при этом может различаться: например, в одном Dataset `forest` может иметь ID 2, а в другом ID 5.

Для обучения и особенно для blending это опасно. Поэтому приводим аудиометки к порядку классов из `image/train`.

### Разбор проверки и приведения меток к единому порядку

`dataset.features["label"].names` возвращает список названий классов в том порядке, который использует конкретный Dataset. Числовая метка `0` соответствует первому имени, `1` — второму и так далее.

`set(image_labels) != set(audio_labels)` сравнивает именно наборы имён, не учитывая порядок. Если наборы разные, `raise ValueError` останавливает ноутбук: модель изображения и аудио должны предсказывать одинаковые 13 классов.

`label2id` и `id2label` — два словаря:
- `label2id`: название класса → целочисленный ID;
- `id2label`: ID → название класса.

Порядок ID выбираем по `image/train`. Если порядок `audio/train` другой, функция `remap_audio_label(old_label)`:
1. получает старый ID аудио-класса;
2. находит его название через `old_audio_labels`;
3. возвращает новый ID по общему `label2id`.

`audio_train_all.map(...)` применяет функцию ко всем строкам Dataset. Аргумент `input_columns=["label"]` передаёт функции только метку, а не декодирует аудиофайл повторно. `cast_column("label", ClassLabel(names=labels))` задаёт столбцу тип именованной метки с общим порядком классов.

`len(dataset)` считает количество примеров. `warnings.warn(...)` сообщает, если найдено не 13 классов, но не останавливает выполнение. Проверки пустых наборов через `raise ValueError` уже останавливают выполнение, потому что пустую выборку нельзя обучать или предсказывать.

In [ ]:
image_labels = image_train_all.features["label"].names
audio_labels = audio_train_all.features["label"].names

if set(image_labels) != set(audio_labels):
    raise ValueError(
        "В image/train и audio/train разные наборы классов.\n"
        f"Только в image: {sorted(set(image_labels) - set(audio_labels))}\n"
        f"Только в audio: {sorted(set(audio_labels) - set(image_labels))}"
    )

labels = image_labels
label2id = {name: idx for idx, name in enumerate(labels)}
id2label = {idx: name for idx, name in enumerate(labels)}

if audio_labels != image_labels:
    print("Порядок аудиоклассов отличается. Перенумеровываем метки audio/train.")
    old_audio_labels = audio_labels

    def remap_audio_label(old_label):
        old_name = old_audio_labels[int(old_label)]
        return {"label": label2id[old_name]}

    # Передаём map только колонку label, чтобы не декодировать OGG-файлы повторно.
    audio_train_all = audio_train_all.map(
        remap_audio_label,
        input_columns=["label"],
    )
    audio_train_all = audio_train_all.cast_column(
        "label", ClassLabel(names=labels)
    )

print("Количество классов:", len(labels))
print("Названия классов:", labels)
print("Image train:", len(image_train_all))
print("Audio train:", len(audio_train_all))
print("Image val:", len(image_val))
print("Audio val:", len(audio_val))

if len(labels) != 13:
    warnings.warn(f"Ожидалось 13 классов, найдено {len(labels)}.")
if len(image_train_all) == 0 or len(audio_train_all) == 0:
    raise ValueError("Одна из обучающих выборок пустая.")
if len(image_val) == 0 or len(audio_val) == 0:
    raise ValueError("Одна из val-выборок пустая.")

## 6. Анализ дисбаланса классов

Количество изображений и аудиозаписей **не обязано совпадать**. ViT обучается на изображениях, AST — на аудио. Нас интересует баланс внутри каждой модальности отдельно.

Например, 4000 изображений класса `forest` и 700 аудиозаписей `forest` — это не ошибка. Это просто разные размеры наборов данных.

Сначала обучаем базовые модели на исходном распределении. Если после оценки редкие классы действительно распознаются хуже, можно попробовать `USE_CLASS_WEIGHTS=True` и сравнить результаты.

### Разбор подсчёта количества примеров по классам

`Counter(image_train_all["label"])` считает, сколько раз встречается каждый ID класса в изображениях. Аналогично считается аудио. Здесь используется исходный размеченный train до разделения — мы хотим увидеть реальный исходный баланс.

`pd.DataFrame({...})` создаёт таблицу: одна строка на класс, отдельные столбцы для количества изображений и аудио. Вызов `counts.get(i, 0)` получает число объектов класса `i`; если ID отсутствует, возвращается 0 вместо ошибки.

Проценты рассчитываются как:

\[
\text{доля класса, %}=100\cdot\frac{\text{число объектов класса}}{\text{число всех объектов}}
\]

`.round(2)` округляет проценты до двух знаков после запятой. `display(balance_df)` красиво показывает таблицу в Jupyter.

В цикле `counts > 0` оставляет только классы с положительным количеством файлов. `positive.max() / positive.min()` показывает отношение самого многочисленного класса к самому редкому. Например, значение 20 означает, что в самом большом классе в 20 раз больше примеров, чем в самом маленьком.

Важно: дисбаланс изображения и аудио анализируется отдельно. Не требуется, чтобы число JPG и OGG одного класса совпадало.

In [ ]:
image_counts = Counter(image_train_all["label"])
audio_counts = Counter(audio_train_all["label"])

balance_df = pd.DataFrame({
    "class": labels,
    "image_count": [image_counts.get(i, 0) for i in range(len(labels))],
    "audio_count": [audio_counts.get(i, 0) for i in range(len(labels))],
})

balance_df["image_percent"] = (
    100 * balance_df["image_count"] / len(image_train_all)
).round(2)
balance_df["audio_percent"] = (
    100 * balance_df["audio_count"] / len(audio_train_all)
).round(2)

display(balance_df)

for modality, counts in [
    ("image", balance_df["image_count"].to_numpy()),
    ("audio", balance_df["audio_count"].to_numpy()),
]:
    positive = counts[counts > 0]
    if len(positive):
        print(f"{modality}: отношение max/min = {positive.max() / positive.min():.2f}")

### Разбор построения двух графиков

`plt.subplots(1, 2, figsize=(16, 5))` создаёт одну фигуру с двумя областями для графиков: 1 строка, 2 столбца; `figsize` задаёт размер в дюймах. `fig` — вся фигура, `axes` — массив отдельных областей.

`axes[0].bar(labels, balance_df["image_count"])` строит столбчатую диаграмму количества изображений по классам. `axes[1].bar(...)` делает то же для аудио.

- `set_title()` задаёт заголовок;
- `set_ylabel()` подписывает вертикальную ось;
- `tick_params(axis="x", rotation=75)` поворачивает подписи классов, чтобы они не налезали друг на друга;
- `plt.tight_layout()` автоматически подбирает отступы;
- `plt.show()` отображает графики.

Смысл графиков — быстро увидеть, есть ли редкие классы и одинаков ли дисбаланс в двух модальностях.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(labels, balance_df["image_count"])
axes[0].set_title("Image train: количество изображений по классам")
axes[0].set_ylabel("Количество файлов")
axes[0].tick_params(axis="x", rotation=75)

axes[1].bar(labels, balance_df["audio_count"])
axes[1].set_title("Audio train: количество аудиозаписей по классам")
axes[1].set_ylabel("Количество файлов")
axes[1].tick_params(axis="x", rotation=75)

plt.tight_layout()
plt.show()

### Как поступать при дисбалансе

1. **Базовый запуск без балансировки.** Не удаляем многочисленные файлы и не дублируем редкие.
2. **Смотрим Macro F1 и отчёт по классам.** Высокая Accuracy сама по себе не гарантирует хорошего распознавания редких классов.
3. **Пробуем веса классов**, если редкие классы действительно дают слабый Recall/F1. Веса рассчитываются отдельно для ViT и AST.
4. **Сравниваем эксперименты на одной и той же внутренней выборке.** Не включаем одновременно class weights и oversampling без отдельного эксперимента.

Формула весов:

\[
w_c = \left(\frac{N}{K \cdot n_c}\right)^p
\]

где `N` — число обучающих объектов, `K` — число классов, `n_c` — число объектов класса, `p` — сила компенсации. В ноутбуке `p=0.5` означает более мягкое взвешивание, чем полная обратная частота.

## 7. Проверка, можно ли сопоставить JPG и OGG

Для настоящего мультимодального blending обе модели должны предсказывать **один и тот же объект**. В качестве предварительного идентификатора используем сочетание `(название класса, имя файла без расширения)` для размеченного `train`.

Это предположение нужно подтвердить: одинаковые имена файлов ещё не доказывают, что они записаны для одного и того же участка.

### Разбор функций сопоставления файлов

**`get_labeled_file_records(dataset, column, feature_type)`** собирает метаданные о файлах, не загружая пиксели или звуковые волны в память.

Аргументы:
- `dataset` — размеченный Dataset;
- `column` — имя столбца (`"image"` или `"audio"`);
- `feature_type` — `HFImage` или `Audio`, чтобы переключить соответствующий столбец в режим `decode=False`.

`dataset.cast_column(column, feature_type(decode=False))` меняет способ чтения столбца: вместо декодированного изображения/аудио можно получить путь к файлу. `range(len(raw))` перебирает индексы строк; `raw[idx]` получает одну строку Dataset.

`item.get("path")` берёт путь, если элемент представлен словарём. `Path(path).stem` — имя без расширения (`001` из `001.jpg`), `Path(path).name` — имя с расширением. Ключ `(class_name, stem)` учитывает и класс, и ID файла: это защищает от случайного совпадения одинакового имени в разных классах.

**`unique_record_map(records)`** группирует записи по ключу. `defaultdict(list)` создаёт список для нового ключа, `grouped[key].append(record)` добавляет запись в группу. Возвращаются два словаря: уникальные ключи и дублирующиеся ключи. Дубли не используются для пар, потому что невозможно однозначно выбрать соответствующий файл.

`set(image_train_map) & set(audio_train_map)` находит ключи, существующие в обеих модальностях. Это предварительные пары JPG/OGG. Само совпадение имени — техническое предположение; нужно подтвердить, что оно действительно означает один участок местности.

In [ ]:
def get_labeled_file_records(dataset, column, feature_type):
    """Читает пути и метки, не декодируя сами изображения/аудио."""
    raw = dataset.cast_column(column, feature_type(decode=False))
    records = []

    for idx in range(len(raw)):
        row = raw[idx]
        item = row[column]
        path = item.get("path") if isinstance(item, dict) else None
        if not path:
            continue

        label_id = int(row["label"])
        class_name = labels[label_id]
        stem = Path(path).stem
        records.append({
            "index": idx,
            "path": path,
            "filename": Path(path).name,
            "stem": stem,
            "class_name": class_name,
            "label": label_id,
            "key": (class_name, stem),
        })
    return records


def unique_record_map(records):
    grouped = defaultdict(list)
    for record in records:
        grouped[record["key"]].append(record)

    unique = {key: rows[0] for key, rows in grouped.items() if len(rows) == 1}
    duplicates = {key: rows for key, rows in grouped.items() if len(rows) > 1}
    return unique, duplicates

image_train_records = get_labeled_file_records(
    image_train_all, "image", HFImage
)
audio_train_records = get_labeled_file_records(
    audio_train_all, "audio", Audio
)

image_train_map, image_train_duplicates = unique_record_map(image_train_records)
audio_train_map, audio_train_duplicates = unique_record_map(audio_train_records)

paired_train_keys = sorted(set(image_train_map) & set(audio_train_map))
pair_counts = Counter(class_name for class_name, _ in paired_train_keys)

print("Уникальных image train файлов:", len(image_train_map))
print("Уникальных audio train файлов:", len(audio_train_map))
print("Совпадающих train-пар:", len(paired_train_keys))
print("Повторяющихся ключей image:", len(image_train_duplicates))
print("Повторяющихся ключей audio:", len(audio_train_duplicates))

display(pd.DataFrame({
    "class": labels,
    "paired_train_count": [pair_counts.get(name, 0) for name in labels],
}))

## 8. Создаём внутренние train/validation выборки

В папках `val` нет меток, поэтому для оценки качества выделяем часть размеченных данных из `train`.

**Предпочтительный сценарий:** если у каждой пары JPG/OGG достаточно совпадающих идентификаторов, выделяем общую парную validation. Одни и те же объекты исключаются из обучения обеих моделей и затем используются для проверки ViT, AST и blending.

**Запасной сценарий:** если надёжных пар мало, разделяем image и audio независимо. В этом случае метрики моделей можно смотреть отдельно, но нельзя честно подбирать коэффициент blending по разным наборам объектов.

Функция ниже старается сохранить каждый класс в обучающей части. Если в классе только один пример, он остаётся в train, а в validation для него примеров не будет.

### Разбор разделения на train и validation

**Функция `stratified_indices(labels_array, n_classes, test_fraction=0.2, seed=42)`** возвращает два списка индексов: для обучения и для внутренней проверки.

- `labels_array` — числовая метка каждого объекта;
- `n_classes` — сколько классов ожидается;
- `test_fraction` — доля, которую хотим отложить для оценки;
- `seed` — фиксирует случайное перемешивание.

Внутри:
- `np.asarray(..., dtype=int)` превращает метки в целочисленный массив NumPy;
- `np.random.default_rng(seed)` создаёт генератор случайных чисел с фиксированным seed;
- `np.flatnonzero(labels_array == class_id)` находит индексы всех объектов конкретного класса;
- `rng.shuffle(...)` случайно перемешивает индексы внутри класса;
- `extend()` добавляет индексы в общий список;
- `round()` рассчитывает число объектов в validation; `max(1, ...)` старается взять хотя бы один, а `min(..., len(class_indices)-1)` оставляет хотя бы один пример в train;
- `sorted()` возвращает упорядоченные индексы.

Если в классе только один объект, он остаётся в train: оценить качество класса по одному и тому же примеру, одновременно использовав его для обучения, было бы некорректно.

Дальше проверяется, есть ли достаточно общих пар во всех классах. Если да, один и тот же набор парных ID исключается из обучения обеих моделей и становится общей validation. Если нет, image и audio разбиваются независимо, и смешивать их validation-предсказания для подбора blending нельзя.

`Dataset.select(indices)` создаёт поднабор по указанным индексам. Благодаря этому обе модальности получают одни и те же парные объекты в validation, а соответствующие объекты не попадают в train ни одной модели.

In [ ]:
def stratified_indices(labels_array, n_classes, test_fraction=0.2, seed=42):
    """Стратифицированное разделение индексов с защитой от singleton-классов."""
    labels_array = np.asarray(labels_array, dtype=int)
    rng = np.random.default_rng(seed)
    train_indices, eval_indices = [], []

    for class_id in range(n_classes):
        class_indices = np.flatnonzero(labels_array == class_id)
        rng.shuffle(class_indices)

        if len(class_indices) == 0:
            raise ValueError(f"В исходных данных отсутствует класс ID={class_id}.")
        if len(class_indices) == 1:
            train_indices.extend(class_indices.tolist())
            continue

        n_eval = max(1, int(round(len(class_indices) * test_fraction)))
        n_eval = min(n_eval, len(class_indices) - 1)
        eval_indices.extend(class_indices[:n_eval].tolist())
        train_indices.extend(class_indices[n_eval:].tolist())

    return sorted(train_indices), sorted(eval_indices)


MIN_PAIRED_PER_CLASS_FOR_BLEND = 10
paired_counts_by_class = [pair_counts.get(name, 0) for name in labels]
can_make_paired_eval = all(
    count >= MIN_PAIRED_PER_CLASS_FOR_BLEND
    for count in paired_counts_by_class
)

if can_make_paired_eval:
    pair_y = np.array([label2id[class_name] for class_name, _ in paired_train_keys])
    _, pair_eval_indices = stratified_indices(
        pair_y, n_classes=len(labels), test_fraction=0.2, seed=SEED
    )
    ordered_eval_keys = [paired_train_keys[i] for i in pair_eval_indices]

    image_eval_indices = [image_train_map[key]["index"] for key in ordered_eval_keys]
    audio_eval_indices = [audio_train_map[key]["index"] for key in ordered_eval_keys]

    image_eval_index_set = set(image_eval_indices)
    audio_eval_index_set = set(audio_eval_indices)

    image_train_indices = [i for i in range(len(image_train_all)) if i not in image_eval_index_set]
    audio_train_indices = [i for i in range(len(audio_train_all)) if i not in audio_eval_index_set]

    image_train = image_train_all.select(image_train_indices)
    image_eval = image_train_all.select(image_eval_indices)
    audio_train = audio_train_all.select(audio_train_indices)
    audio_eval = audio_train_all.select(audio_eval_indices)

    BLENDING_EVAL_AVAILABLE = True
    print("Создана общая парная validation для image и audio.")
    print("Количество пар в validation:", len(ordered_eval_keys))
else:
    print(
        "Совпадающих train-пар недостаточно для общей validation. "
        "Разделяем модальности независимо."
    )

    image_train_idx, image_eval_idx = stratified_indices(
        image_train_all["label"], len(labels), seed=SEED
    )
    audio_train_idx, audio_eval_idx = stratified_indices(
        audio_train_all["label"], len(labels), seed=SEED
    )

    image_train = image_train_all.select(image_train_idx)
    image_eval = image_train_all.select(image_eval_idx)
    audio_train = audio_train_all.select(audio_train_idx)
    audio_eval = audio_train_all.select(audio_eval_idx)

    BLENDING_EVAL_AVAILABLE = False

print("Image train / validation:", len(image_train), "/", len(image_eval))
print("Audio train / validation:", len(audio_train), "/", len(audio_eval))

## 9. Метрики и необязательное взвешивание классов

Будем сохранять две основные метрики:

- **Accuracy** — доля правильных предсказаний.
- **Macro F1** — F1 считается отдельно по каждому классу, затем классы усредняются с одинаковым весом.

Для выбора модели при дисбалансе ориентируемся прежде всего на Macro F1, но также смотрим Accuracy, Precision и Recall по каждому классу.

### Разбор метрик, весов классов и собственного Trainer

**`compute_metrics(eval_pred)`** вызывается Trainer после оценки. `eval_pred.predictions` — logits модели; иногда библиотека возвращает их внутри tuple, поэтому проверяется `isinstance`. `np.argmax(..., axis=1)` выбирает ID класса с максимальным logit для каждой строки. `eval_pred.label_ids` — правильные метки.

- `accuracy_score(y_true, y_pred)` считает долю верных ответов.
- `f1_score(..., average="macro")` считает F1 отдельно по классам и усредняет классы с одинаковым весом. `labels=np.arange(len(labels))` фиксирует полный список ID классов; `zero_division=0` задаёт 0, если для класса Precision/Recall не может быть вычислена из-за отсутствия предсказаний или примеров.

**`calculate_class_weights(dataset, power=0.5)`** рассчитывает вес каждого класса только по обучающей части. `np.bincount` считает объекты для каждого ID; `minlength` сохраняет место даже для отсутствующих ID. Если класс отсутствует, функция останавливается с `ValueError`, потому что вес для него нельзя надёжно рассчитать. Формула `(N / (K * n_c)) ** power` повышает вес редких классов; `power=0.5` делает это мягче, чем полная обратная частота. Деление на `weights.mean()` приводит средний вес к 1. Возвращаются тензор весов и исходные количества.

**`ClassWeightedTrainer`** наследуется от стандартного `Trainer` и меняет функцию потерь:
- `__init__` передаёт остальные аргументы родительскому `Trainer` через `super().__init__()` и сохраняет веса классов;
- `compute_loss(...)` получает батч `inputs`, отделяет `labels`, выполняет прямой проход `model(**model_inputs)`, переносит веса на то же устройство и в тот же тип данных, что и logits, затем считает `F.cross_entropy(..., weight=weights)`.
- `return_outputs=True` просит вернуть не только loss, но и выходы модели; это предусмотрено интерфейсом Trainer.

**`build_trainer(...)`** собирает общие аргументы один раз. `DefaultDataCollator()` объединяет отдельные примеры в батч; `train_dataset` и `eval_dataset` задают данные для оптимизации и оценки; `processing_class` хранит соответствующий image/audio processor; `compute_metrics` подключает метрики. Если `USE_CLASS_WEIGHTS=True`, создаётся `ClassWeightedTrainer`; иначе обычный `Trainer`. Так код обучения ViT и AST остаётся единообразным.

In [ ]:
def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    y_pred = np.argmax(logits, axis=1)
    y_true = eval_pred.label_ids

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=np.arange(len(labels)),
            average="macro",
            zero_division=0,
        ),
    }


def calculate_class_weights(dataset, power=0.5):
    y = np.asarray(dataset["label"], dtype=int)
    counts = np.bincount(y, minlength=len(labels)).astype(np.float64)
    if np.any(counts == 0):
        missing = [labels[i] for i, count in enumerate(counts) if count == 0]
        raise ValueError(f"В train отсутствуют классы: {missing}")

    weights = (counts.sum() / (len(labels) * counts)) ** power
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32), counts


class ClassWeightedTrainer(Trainer):
    """Trainer с взвешенной CrossEntropyLoss для редких классов."""
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs,
    ):
        labels_batch = inputs["labels"]
        model_inputs = {key: value for key, value in inputs.items() if key != "labels"}
        outputs = model(**model_inputs)
        weights = self.class_weights.to(
            device=outputs.logits.device,
            dtype=outputs.logits.dtype,
        )
        loss = F.cross_entropy(outputs.logits, labels_batch, weight=weights)
        return (loss, outputs) if return_outputs else loss


def build_trainer(model, args, train_ds, eval_ds, processing_class, class_weights=None):
    common = dict(
        model=model,
        args=args,
        data_collator=DefaultDataCollator(),
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        processing_class=processing_class,
        compute_metrics=compute_metrics,
    )

    if USE_CLASS_WEIGHTS:
        return ClassWeightedTrainer(class_weights=class_weights, **common)
    return Trainer(**common)

## 10. Визуальная модель: ViT

Используем предварительно обученную модель `google/vit-base-patch16-224-in21k`. Она уже обучалась извлекать визуальные признаки, а её классификационная голова будет заменена на новую — с 13 выходами.

Случайная обрезка применяется только при обучении. Для validation и итогового `val` используем фиксированные Resize и CenterCrop, чтобы предсказания были воспроизводимыми.

Для спутниковых изображений берём более мягкий диапазон случайной обрезки `scale=(0.7, 1.0)`: слишком сильная обрезка иногда может удалить важный контекст местности.

### Разбор Image Processor и преобразований

`AutoImageProcessor.from_pretrained(IMAGE_CHECKPOINT)` загружает обработчик, соответствующий выбранной ViT. Он содержит нужные размер изображения, средние значения каналов (`image_mean`) и стандартные отклонения (`image_std`). Это важно: параметры предобработки должны соответствовать модели, на которой она предварительно обучалась.

`image_processor.size` — словарь с требуемыми размерами. Код проверяет, указаны ли отдельно `height` и `width`; если да, использует пару значений. Иначе использует `shortest_edge` (обычно 224).

`Normalize(mean=..., std=...)` применяет к каждому RGB-каналу формулу `(x - mean) / std`.

`Compose([...])` объединяет несколько преобразований в последовательность:
- при обучении `RandomResizedCrop(..., scale=(0.7, 1.0))` случайно выбирает область изображения, занимающую 70–100% исходной площади, и приводит её к целевому размеру;
- `ToTensor()` превращает PIL-изображение в тензор PyTorch формата `[C, H, W]` и масштабирует пиксели в диапазон `[0, 1]`;
- `Normalize` нормализует каналы.

Для оценки используются `Resize` и `CenterCrop` — детерминированные операции без случайной обрезки. Поэтому одно и то же изображение при повторной оценке будет обработано одинаково.

In [ ]:
IMAGE_CHECKPOINT = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(IMAGE_CHECKPOINT)

size_config = image_processor.size
if "height" in size_config and "width" in size_config:
    image_crop_size = (size_config["height"], size_config["width"])
    image_resize_size = image_crop_size
else:
    image_crop_size = size_config.get("shortest_edge", 224)
    image_resize_size = image_crop_size

image_normalize = Normalize(
    mean=image_processor.image_mean,
    std=image_processor.image_std,
)

image_train_transforms = Compose([
    RandomResizedCrop(image_crop_size, scale=(0.7, 1.0)),
    ToTensor(),
    image_normalize,
])

image_eval_transforms = Compose([
    Resize(image_resize_size),
    CenterCrop(image_crop_size),
    ToTensor(),
    image_normalize,
])

print("Размер входа:", image_crop_size)
print("Mean:", image_processor.image_mean)
print("Std:", image_processor.image_std)

### 10.1. Преобразуем изображения в тензоры

`with_transform()` применяет функцию при чтении примеров. Это удобно, потому что мы не создаём заранее отдельную копию всех обработанных изображений.

Каждое изображение превращается в `pixel_values` с формой `[3, H, W]`:

- `3` — RGB-каналы;
- `H` — высота;
- `W` — ширина.

### Разбор функций преобразования изображений

**`_as_list(value)`** — вспомогательная функция. `with_transform()` в зависимости от способа чтения может передать функции один пример или пачку примеров. Эта функция приводит оба варианта к списку и возвращает флаг `is_single`, который сообщает, был ли исходный вход одиночным.

**`image_train_transform(examples)`** применяет обучающие преобразования:
1. берёт поле `examples["image"]`;
2. приводит одно изображение или список изображений к общему формату через `_as_list`;
3. `img.convert("RGB")` гарантирует три цветовых канала;
4. `image_train_transforms(img)` выполняет случайную обрезку, перевод в тензор и нормализацию;
5. создаёт поле `pixel_values`, которое ожидает ViT;
6. `del examples["image"]` убирает исходный PIL-объект из возвращаемого примера, чтобы Trainer передавал модели уже подготовленный тензор;
7. `return examples` возвращает изменённый словарь.

**`image_eval_transform(examples)`** делает то же самое, но применяет фиксированные `image_eval_transforms` без случайной аугментации.

`dataset.with_transform(function)` не преобразует и не сохраняет весь датасет заранее. Он назначает функцию, которая вызывается при чтении элементов. Поэтому случайная аугментация может немного меняться между обращениями.

`image_train_ds[0]` извлекает один пример и тем самым проверяет, что функция действительно создала `pixel_values`. `.shape` показывает форму тензора: `[3, H, W]`. `labels[int(...)]` переводит числовой ID в название класса для понятного вывода.

In [ ]:
def _as_list(value):
    if isinstance(value, (list, tuple)):
        return list(value), False
    return [value], True


def image_train_transform(examples):
    images, is_single = _as_list(examples["image"])
    pixel_values = [
        image_train_transforms(img.convert("RGB"))
        for img in images
    ]
    examples["pixel_values"] = pixel_values[0] if is_single else pixel_values
    del examples["image"]
    return examples


def image_eval_transform(examples):
    images, is_single = _as_list(examples["image"])
    pixel_values = [
        image_eval_transforms(img.convert("RGB"))
        for img in images
    ]
    examples["pixel_values"] = pixel_values[0] if is_single else pixel_values
    del examples["image"]
    return examples


image_train_ds = image_train.with_transform(image_train_transform)
image_eval_ds = image_eval.with_transform(image_eval_transform)
image_val_ds = image_val.with_transform(image_eval_transform)

image_sample = image_train_ds[0]
print("Форма pixel_values:", image_sample["pixel_values"].shape)
print("ID класса:", image_sample["label"], "->", labels[int(image_sample["label"])])

### 10.2. Инициализация ViT и Trainer

`remove_unused_columns=False` обязателен для такого варианта `with_transform()`: Trainer не должен удалить исходную колонку `image` до того, как наша функция преобразует её в `pixel_values`.

`per_device_eval_batch_size` также будет использоваться при `Trainer.predict()`: именно он определяет, сколько validation/val-примеров модель обрабатывает за один шаг на устройстве.

### Разбор модели ViT и TrainingArguments

**Загрузка модели:** `AutoModelForImageClassification.from_pretrained(...)` загружает архитектуру и предварительно обученные веса.
- `IMAGE_CHECKPOINT` — имя исходной модели ViT;
- `num_labels=len(labels)` — размер классификационной головы, здесь 13 выходов;
- `id2label` и `label2id` задают соответствие ID и названий;
- `ignore_mismatched_sizes=True` разрешает переинициализировать классификационную голову, если размер исходного выхода не совпадает с 13 классами.

**Настройки `TrainingArguments`:**
- `output_dir` — папка для чекпойнтов и конфигурации;
- `eval_strategy="epoch"` — оценивать модель после каждой эпохи;
- `save_strategy="epoch"` — сохранять чекпойнт после каждой эпохи;
- `learning_rate` — шаг оптимизатора, определяющий величину обновления весов;
- `per_device_train_batch_size` / `per_device_eval_batch_size` — число примеров на одном устройстве при обучении/оценке;
- `gradient_accumulation_steps` — число батчей, по которым накапливаются градиенты перед обновлением весов;
- `num_train_epochs` — число проходов по train;
- `warmup_ratio` — доля начальных шагов, на протяжении которых learning rate постепенно разогревается;
- `logging_steps` — как часто записывать значения обучения;
- `load_best_model_at_end=True` — после завершения вернуть лучший сохранённый чекпойнт;
- `metric_for_best_model="macro_f1"`, `greater_is_better=True` — выбирать checkpoint по наибольшему Macro F1;
- `save_total_limit=2` — хранить ограниченное число checkpoint-файлов;
- `remove_unused_columns=False` — не удалять исходные поля до вызова `with_transform`, иначе `image` может исчезнуть раньше создания `pixel_values`;
- `fp16` включает смешанную точность на CUDA для экономии памяти; на CPU выключается;
- `gradient_checkpointing` снижает потребление памяти ценой дополнительных вычислений;
- `eval_accumulation_steps` постепенно переносит результаты оценки в CPU;
- `dataloader_num_workers=0` отключает отдельные процессы загрузки данных — это часто проще для Colab и трансформаций;
- `report_to="none"` отключает внешние системы логирования;
- `seed` задаёт seed обучения.

Если включены `USE_CLASS_WEIGHTS`, `calculate_class_weights(image_train, ...)` считает веса именно по train-части, не по validation. Затем `build_trainer(...)` создаёт Trainer с нужной функцией потерь. `image_train_ds` и `image_eval_ds` — уже Dataset с назначенными преобразованиями.

In [ ]:
image_model = AutoModelForImageClassification.from_pretrained(
    IMAGE_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

image_training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "vit_location_model"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=IMAGE_BATCH_SIZE,
    per_device_eval_batch_size=IMAGE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=IMAGE_GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=0.1,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_accumulation_steps=4,
    dataloader_num_workers=0,
    report_to="none",
    seed=SEED,
)

image_weights = None
if USE_CLASS_WEIGHTS:
    image_weights, image_train_class_counts = calculate_class_weights(
        image_train, CLASS_WEIGHT_POWER
    )
    display(pd.DataFrame({"class": labels, "count": image_train_class_counts, "weight": image_weights.numpy()}))

image_trainer = build_trainer(
    model=image_model,
    args=image_training_args,
    train_ds=image_train_ds,
    eval_ds=image_eval_ds,
    processing_class=image_processor,
    class_weights=image_weights,
)

print("ViT Trainer готов.")

### 10.3. Обучение ViT

`trainer.train()` запускает полный цикл обучения: прямой проход, расчёт функции потерь, обратное распространение ошибки, обновление весов и оценку в конце каждой эпохи.

Благодаря `load_best_model_at_end=True` после обучения Trainer загрузит лучший checkpoint по `eval_macro_f1`.

### Разбор запуска обучения и сохранения ViT

`image_trainer.train()` запускает цикл обучения: загрузка батча → прямой проход модели → вычисление loss → обратное распространение градиентов → обновление весов. Возвращаемое значение сохраняется в `image_train_result` и содержит итоговую информацию о ходе обучения.

`image_trainer.evaluate()` отдельно запускает оценку на `eval_dataset`, который был передан в Trainer. Результаты сохраняются в словарь `image_metrics`; в нём будут, например, `eval_loss`, `eval_accuracy` и `eval_macro_f1`.

`image_trainer.save_model(path)` сохраняет лучшую текущую модель и конфигурацию в папку. `image_processor.save_pretrained(path)` сохраняет настройки обработчика изображений туда же. Это важно: для последующего инференса нужны и веса модели, и согласованный с ней processor.

`str(OUTPUT_DIR / "vit_location_model")` превращает объект `Path` в строку, которую принимает метод сохранения.

In [ ]:
image_train_result = image_trainer.train()
image_metrics = image_trainer.evaluate()

print("Метрики ViT:")
print(image_metrics)

image_model_dir = OUTPUT_DIR / "vit_location_model"
image_trainer.save_model(str(image_model_dir))
image_processor.save_pretrained(str(image_model_dir))

### 10.4. Подробная оценка ViT

`Trainer.predict()` здесь применяется к размеченной внутренней validation. Поэтому помимо logits мы получим `label_ids` — настоящие метки, и сможем построить classification report и confusion matrix.

### Разбор детальной оценки ViT

`image_trainer.predict(image_eval_ds)` выполняет прямой проход по всей внутренней validation и возвращает объект `PredictionOutput`.

- `.predictions` — logits модели: оценки по всем 13 классам для каждого изображения;
- иногда `.predictions` является tuple, поэтому код при необходимости берёт первый элемент;
- `.label_ids` — истинные числовые метки из размеченной validation;
- `np.argmax(..., axis=1)` превращает logits в ID самого вероятного класса.

`classification_report(...)` рассчитывает по каждому классу Precision, Recall, F1 и Support (число истинных примеров). `labels=np.arange(len(labels))` задаёт все ID, `target_names=labels` подписывает классы именами, `zero_division=0` избегает деления на ноль, а `output_dict=True` возвращает результат как словарь, который удобно превратить в DataFrame.

`confusion_matrix(y_true, y_pred, labels=...)` строит матрицу ошибок: строки — истинные классы, столбцы — предсказанные. Диагональ содержит правильные ответы, вне диагонали — ошибки между классами.

`plt.subplots()` создаёт область для графика; `ConfusionMatrixDisplay(...).plot()` визуализирует матрицу; `xticks_rotation=90` поворачивает длинные подписи; `colorbar=False` скрывает цветовую шкалу; `tight_layout()` подгоняет поля; `show()` выводит график.

In [ ]:
image_eval_output = image_trainer.predict(image_eval_ds)
image_eval_logits = image_eval_output.predictions
if isinstance(image_eval_logits, tuple):
    image_eval_logits = image_eval_logits[0]
image_eval_true = image_eval_output.label_ids
image_eval_pred = np.argmax(image_eval_logits, axis=1)

image_report = classification_report(
    image_eval_true,
    image_eval_pred,
    labels=np.arange(len(labels)),
    target_names=labels,
    zero_division=0,
    output_dict=True,
)
display(pd.DataFrame(image_report).T)

image_cm = confusion_matrix(
    image_eval_true,
    image_eval_pred,
    labels=np.arange(len(labels)),
)
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(image_cm, display_labels=labels).plot(
    ax=ax, xticks_rotation=90, colorbar=False
)
ax.set_title("ViT — confusion matrix")
plt.tight_layout()
plt.show()

### 10.5. Освобождаем GPU перед обучением AST

ViT и AST не стоит одновременно держать в видеопамяти, особенно на GPU с небольшим объёмом памяти. После сохранения и оценки ViT переносим её на CPU и освобождаем состояние оптимизатора. Позже вернём модель на устройство для инференса.

### Разбор освобождения видеопамяти

**`move_trainer_to_cpu(trainer)`** переносит веса модели на CPU, чтобы освободить GPU для следующей модели.

- `trainer.model.to("cpu")` перемещает параметры модели на центральный процессор;
- `trainer.optimizer = None` и `trainer.lr_scheduler = None` удаляют ссылки на оптимизатор и планировщик, которые могли хранить дополнительные тензоры и состояние обучения;
- `gc.collect()` запускает сборщик мусора Python;
- `torch.cuda.empty_cache()` возвращает неиспользуемые блоки CUDA-кэша системе. Эта команда не может освободить память, которая всё ещё занята живыми тензорами.

После этого ViT остаётся обученной и сохранённой, но находится в RAM, а не в VRAM. Позже функция используется и для AST.

In [ ]:
def move_trainer_to_cpu(trainer):
    trainer.model.to("cpu")
    trainer.optimizer = None
    trainer.lr_scheduler = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


move_trainer_to_cpu(image_trainer)
print("ViT перенесена на CPU; память GPU освобождена.")

## 11. Аудиомодель: AST (Audio Spectrogram Transformer)

Для аудио используем `MIT/ast-finetuned-audioset-10-10-0.4593`. AST преобразует звуковой сигнал в спектральное представление и обрабатывает его Transformer-архитектурой.

Модель предварительно обучена на AudioSet, а её классификационная голова будет переинициализирована под наши 13 классов. Предупреждение о заново инициализированных весах классификационной головы при загрузке ожидаемо.

Для данного checkpoint целевая частота дискретизации — 16 кГц, а ожидаемая последовательность спектральных кадров — 1024. Проверяем значения у загруженного Feature Extractor.

### Разбор загрузки аудиопроцессора и настроек сигнала

`AutoFeatureExtractor.from_pretrained(AUDIO_CHECKPOINT)` загружает обработчик аудио, параметры которого соответствуют выбранному AST-checkpoint.

- `sampling_rate` — целевая частота дискретизации в герцах. Например, 16 000 Гц означает 16 000 отсчётов сигнала в секунду.
- `max_length` — максимальное число временных кадров/элементов спектрального представления, которые ожидает этот обработчик. Значение читается из конфигурации; если параметр отсутствует, код использует 1024.
- `num_mel_bins` — число мел-частотных полос в спектрограмме; значение выводится только для контроля.

`int(...)` приводит конфигурационные значения к обычным целым числам Python.

`dataset.cast_column("audio", Audio(sampling_rate=TARGET_SR, num_channels=1))` перенастраивает аудиостолбец: при чтении сигнал декодируется, приводится к целевой частоте и одному каналу. `cast_column` не меняет исходный OGG-файл на диске; обработка происходит при чтении примеров.

In [ ]:
AUDIO_CHECKPOINT = "MIT/ast-finetuned-audioset-10-10-0.4593"
audio_processor = AutoFeatureExtractor.from_pretrained(AUDIO_CHECKPOINT)

TARGET_SR = int(audio_processor.sampling_rate)
AUDIO_MAX_LENGTH = int(getattr(audio_processor, "max_length", 1024))

print("Целевая частота дискретизации:", TARGET_SR)
print("Максимальная длина спектральной последовательности:", AUDIO_MAX_LENGTH)
print("Количество mel bins:", getattr(audio_processor, "num_mel_bins", "не указано"))

# Декодирование выполняется при обращении к примеру; аудио приводится к 16 кГц и одному каналу.
audio_train = audio_train.cast_column(
    "audio", Audio(sampling_rate=TARGET_SR, num_channels=1)
)
audio_eval = audio_eval.cast_column(
    "audio", Audio(sampling_rate=TARGET_SR, num_channels=1)
)
audio_val_decoded = audio_val.cast_column(
    "audio", Audio(sampling_rate=TARGET_SR, num_channels=1)
)

### 11.1. Декодируем и преобразуем аудио

В современных версиях Datasets при чтении аудио может возвращаться объект `AudioDecoder`. Его метод `get_all_samples()` декодирует аудиосигнал. В более старых версиях мог возвращаться словарь с полями `array` и `sampling_rate`; функция ниже поддерживает оба варианта.

После декодирования:

1. получаем waveform — последовательность амплитуд звука;
2. проверяем число каналов и при необходимости сводим звук к моно;
3. при необходимости дополнительно пересчитываем частоту дискретизации;
4. передаём сигнал AST Feature Extractor, который строит входные спектральные признаки.

### Разбор декодирования и преобразования аудио

**`decode_audio_item(item)`** приводит разные варианты представления аудио к одномерному массиву `float32`.

1. `hasattr(item, "get_all_samples")` проверяет, является ли `item` современным `AudioDecoder`. В этом случае `get_all_samples()` получает декодированные отсчёты, `.data` содержит waveform, а `.sample_rate` — частоту дискретизации.
2. В старом формате Datasets аудио могло быть словарём с `array` и `sampling_rate`. Ветка `isinstance(item, dict)` поддерживает такой формат.
3. Если формат неизвестен, `raise TypeError` останавливает обработку и просит проверить декодирование.
4. `torch.is_tensor()` проверяет, является ли waveform тензором PyTorch. `.detach().cpu().numpy()` отсоединяет его от графа градиентов, переносит на CPU и преобразует в NumPy.
5. `np.asarray(..., dtype=np.float32)` приводит данные к массиву нужного числового типа.
6. `ndim` показывает число измерений. Если сигнал 2D, код пытается распознать, где каналы: `(channels, samples)` или `(samples, channels)`, и усредняет каналы в моно. Если форма неожиданная или массив пустой, выдаётся ошибка.
7. Если частота отличается от `TARGET_SR`, `gcd()` сокращает отношение частот, а `resample_poly()` выполняет полифазное изменение частоты дискретизации. Это меняет число отсчётов так, чтобы длительность звука оставалась примерно прежней. Нельзя просто присвоить сигналу другое число Hz без пересчёта.

**`audio_transform(examples)`** готовит один аудиопример или батч:
- `_as_list()` приводит поле `audio` к списку и запоминает одиночный/пакетный формат;
- `decode_audio_item()` декодирует каждый сигнал;
- `audio_processor(...)` строит входные спектральные признаки AST. `sampling_rate` сообщает фактическую частоту; `max_length` задаёт длину; `truncation=True` обрезает слишком длинные записи; `padding="max_length"` дополняет короткие записи до фиксированной длины; `return_tensors="pt"` просит вернуть PyTorch-тензоры;
- `processed["input_values"]` — подготовленные признаки, которые ожидает AST;
- при одиночном примере берётся первый тензор, при батче сохраняется список тензоров;
- исходный столбец `audio` удаляется, а функция возвращает подготовленный словарь.

`with_transform(audio_transform)` назначает функцию при чтении примеров для train, validation и настоящего `audio/val`. Последняя проверка `audio_train_ds[0]` действительно декодирует и обрабатывает один файл, после чего печатает форму входа и соответствующую метку.

In [ ]:
def decode_audio_item(item):
    """Декодирует AudioDecoder (Datasets 4.x) или старый dict с array."""
    if hasattr(item, "get_all_samples"):
        samples = item.get_all_samples()
        waveform = samples.data
        sample_rate = int(samples.sample_rate)
    elif isinstance(item, dict) and "array" in item:
        waveform = item["array"]
        sample_rate = int(item.get("sampling_rate", TARGET_SR))
    else:
        raise TypeError(
            f"Неизвестный формат аудио: {type(item)}. "
            "Проверь версию datasets и декодирование AudioFolder."
        )

    if torch.is_tensor(waveform):
        waveform = waveform.detach().cpu().numpy()
    waveform = np.asarray(waveform, dtype=np.float32)

    # Audio(..., num_channels=1) обычно уже возвращает один канал.
    if waveform.ndim == 2:
        if waveform.shape[0] <= 8:      # формат (channels, samples)
            waveform = waveform.mean(axis=0)
        elif waveform.shape[1] <= 8:    # формат (samples, channels)
            waveform = waveform.mean(axis=1)
        else:
            raise ValueError(f"Неожиданная форма waveform: {waveform.shape}")
    elif waveform.ndim != 1:
        raise ValueError(f"Ожидался 1D или 2D сигнал, получено: {waveform.shape}")

    if waveform.size == 0:
        raise ValueError("В аудиофайле не найдено звуковых отсчётов.")

    # Дополнительная страховка, если источник вернул другую частоту.
    if sample_rate != TARGET_SR:
        common = gcd(sample_rate, TARGET_SR)
        waveform = resample_poly(
            waveform,
            TARGET_SR // common,
            sample_rate // common,
        ).astype(np.float32)

    return waveform


def audio_transform(examples):
    audio_items, is_single = _as_list(examples["audio"])
    waveforms = [decode_audio_item(item) for item in audio_items]

    processed = audio_processor(
        waveforms,
        sampling_rate=TARGET_SR,
        max_length=AUDIO_MAX_LENGTH,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )

    input_values = processed["input_values"]
    examples["input_values"] = input_values[0] if is_single else list(input_values)
    del examples["audio"]
    return examples


audio_train_ds = audio_train.with_transform(audio_transform)
audio_eval_ds = audio_eval.with_transform(audio_transform)
audio_val_ds = audio_val_decoded.with_transform(audio_transform)

audio_sample = audio_train_ds[0]
print("Форма input_values:", audio_sample["input_values"].shape)
print("ID класса:", audio_sample["label"], "->", labels[int(audio_sample["label"])])

### 11.2. Инициализация AST и Trainer

AST обычно требует больше видеопамяти, чем небольшая CNN. Поэтому начинаем с `batch_size=1`, накопления градиентов по 8 шагов и FP16 на GPU.

Если возникает `CUDA out of memory`, сначала проверь, что ViT уже перенесена на CPU. Затем можно уменьшить batch size (он уже равен 1), отключить gradient checkpointing только после проверки памяти или перейти на более компактную аудиоархитектуру.

### Разбор инициализации AST и параметров Trainer

`AutoModelForAudioClassification.from_pretrained(...)` загружает предварительно обученную AST. `num_labels=len(labels)` задаёт 13 выходов; `id2label`/`label2id` задают общий порядок классов; `ignore_mismatched_sizes=True` разрешает заменить исходную классификационную голову под нашу задачу.

В `TrainingArguments` параметры имеют тот же смысл, что у ViT:
- `output_dir` — папка чекпойнтов AST;
- `eval_strategy` и `save_strategy` — оценка и сохранение после каждой эпохи;
- `learning_rate` — скорость изменения весов;
- `per_device_train_batch_size` и `per_device_eval_batch_size` — число аудиопримеров в одном батче на устройстве;
- `gradient_accumulation_steps` — число батчей накопления градиентов;
- `num_train_epochs` — число эпох;
- `warmup_ratio` — доля warmup-шагов;
- `logging_steps` — частота логирования;
- `load_best_model_at_end`, `metric_for_best_model`, `greater_is_better` — загрузить лучший checkpoint по Macro F1;
- `save_total_limit` — ограничить количество сохранённых checkpoint;
- `remove_unused_columns=False` — сохранить исходную колонку `audio` до того, как `with_transform` создаст `input_values`;
- `fp16` — смешанная точность на CUDA;
- `gradient_checkpointing=True` — экономия памяти с дополнительными вычислениями;
- `eval_accumulation_steps` — периодически переносить результаты оценки с GPU;
- `dataloader_num_workers=0` — загружать данные в основном процессе;
- `report_to="none"` — не подключать внешний логгер;
- `seed` — seed для воспроизводимости.

Если включено взвешивание, веса считаются по `audio_train`, а не по validation. `build_trainer(...)` создаёт стандартный или взвешенный Trainer. В него передаются уже преобразуемые `audio_train_ds` и `audio_eval_ds`, а также `audio_processor`, чтобы сохранить настройки AST.

In [ ]:
audio_model = AutoModelForAudioClassification.from_pretrained(
    AUDIO_CHECKPOINT,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

audio_training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "ast_location_model"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=AUDIO_BATCH_SIZE,
    per_device_eval_batch_size=AUDIO_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=AUDIO_GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=0.1,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    remove_unused_columns=False,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_accumulation_steps=2,
    dataloader_num_workers=0,
    report_to="none",
    seed=SEED,
)

audio_weights = None
if USE_CLASS_WEIGHTS:
    audio_weights, audio_train_class_counts = calculate_class_weights(
        audio_train, CLASS_WEIGHT_POWER
    )
    display(pd.DataFrame({"class": labels, "count": audio_train_class_counts, "weight": audio_weights.numpy()}))

audio_trainer = build_trainer(
    model=audio_model,
    args=audio_training_args,
    train_ds=audio_train_ds,
    eval_ds=audio_eval_ds,
    processing_class=audio_processor,
    class_weights=audio_weights,
)

print("AST Trainer готов.")

### 11.3. Обучение AST

Запуск обучает аудиомодель независимо от ViT. После каждой эпохи Trainer оценивает модель на внутренней размеченной validation. В конце сохраняется checkpoint с лучшим Macro F1.

### Разбор обучения и сохранения AST

`audio_trainer.train()` запускает оптимизацию AST на размеченном `audio_train_ds`. `audio_trainer.evaluate()` рассчитывает loss и метрики на `audio_eval_ds`, не обновляя веса.

`audio_trainer.save_model(...)` сохраняет обученную модель и конфигурацию. `audio_processor.save_pretrained(...)` сохраняет настройки обработки звука. При последующей загрузке AST нужно использовать оба сохранённых компонента, чтобы формат входа совпадал с тем, на котором модель обучалась.

In [ ]:
audio_train_result = audio_trainer.train()
audio_metrics = audio_trainer.evaluate()

print("Метрики AST:")
print(audio_metrics)

audio_model_dir = OUTPUT_DIR / "ast_location_model"
audio_trainer.save_model(str(audio_model_dir))
audio_processor.save_pretrained(str(audio_model_dir))

### 11.4. Подробная оценка AST

Как и для ViT, получаем logits на внутренней validation, сравниваем их с настоящими метками и строим отчёт по каждому классу.

### Разбор детальной оценки AST

`audio_trainer.predict(audio_eval_ds)` получает предсказания по всей внутренней validation. `.predictions` содержит logits AST, `.label_ids` — правильные метки. Проверка tuple нужна для совместимости с разными форматами выхода модели. `np.argmax(..., axis=1)` превращает logits в ID классов.

`classification_report` формирует Precision, Recall, F1 и Support по каждому аудиоклассу. `confusion_matrix` считает, какие классы модель путает между собой. `ConfusionMatrixDisplay(...).plot(...)` отображает эту матрицу; остальные команды Matplotlib задают размер, заголовок, поворот подписей и вывод графика.

Смысл этой ячейки — не только получить одну Accuracy, но и понять, например, какие звуковые классы AST распознаёт хуже других.

In [ ]:
audio_eval_output = audio_trainer.predict(audio_eval_ds)
audio_eval_logits = audio_eval_output.predictions
if isinstance(audio_eval_logits, tuple):
    audio_eval_logits = audio_eval_logits[0]
audio_eval_true = audio_eval_output.label_ids
audio_eval_pred = np.argmax(audio_eval_logits, axis=1)

audio_report = classification_report(
    audio_eval_true,
    audio_eval_pred,
    labels=np.arange(len(labels)),
    target_names=labels,
    zero_division=0,
    output_dict=True,
)
display(pd.DataFrame(audio_report).T)

audio_cm = confusion_matrix(
    audio_eval_true,
    audio_eval_pred,
    labels=np.arange(len(labels)),
)
fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(audio_cm, display_labels=labels).plot(
    ax=ax, xticks_rotation=90, colorbar=False
)
ax.set_title("AST — confusion matrix")
plt.tight_layout()
plt.show()

### 11.5. Освобождаем GPU после AST

Перед итоговым инференсом будем загружать каждую модель на GPU по очереди. Это снижает риск переполнения видеопамяти.

### Разбор вызова очистки памяти

`move_trainer_to_cpu(audio_trainer)` вызывает функцию, определённую в разделе ViT. Модель AST перемещается на CPU, ссылки на оптимизатор и планировщик удаляются, запускается сборщик мусора и очищается свободный CUDA-кэш.

Это нужно перед последующими предсказаниями, чтобы не держать обе крупные модели в видеопамяти одновременно. Веса AST уже сохранены в `OUTPUT_DIR`, поэтому их можно загрузить на GPU снова при необходимости.

In [ ]:
move_trainer_to_cpu(audio_trainer)
print("AST перенесена на CPU; память GPU освобождена.")

## 12. Сравнение моделей

Сравниваем Accuracy и Macro F1. Если была создана общая парная validation, обе модели оценивались на одних и тех же объектах — сравнение более корректно.

Если validation формировалась независимо для image и audio, метрики всё равно полезны для анализа каждой модели, но прямое сравнение не является строгим: наборы объектов отличаются.

### Разбор таблицы сравнения

`pd.DataFrame([...])` собирает по строке на каждую модель. `dict.get("eval_accuracy")` и `dict.get("eval_macro_f1")` достают нужные метрики из словарей, возвращённых `Trainer.evaluate()`; `.get()` не вызывает ошибку, если ключ отсутствует, а вернёт `None`.

`sort_values("macro_f1", ascending=False)` сортирует модели от наибольшего Macro F1 к наименьшему. `display()` выводит таблицу.

Условие `if BLENDING_EVAL_AVAILABLE` поясняет, была ли создана единая парная validation. Если image и audio validation независимы, их метрики можно анализировать отдельно, но нельзя использовать два набора разных объектов для честного выбора коэффициента blending.

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "ViT / satellite images",
        "accuracy": image_metrics.get("eval_accuracy"),
        "macro_f1": image_metrics.get("eval_macro_f1"),
    },
    {
        "model": "AST / audio",
        "accuracy": audio_metrics.get("eval_accuracy"),
        "macro_f1": audio_metrics.get("eval_macro_f1"),
    },
])

display(comparison.sort_values("macro_f1", ascending=False))

if BLENDING_EVAL_AVAILABLE:
    print("Модели оценивались на общей парной validation.")
else:
    print(
        "ВНИМАНИЕ: validation выборки независимые. "
        "Результаты каждой модели можно анализировать отдельно, "
        "но выбирать blending alpha на этих разных выборках нельзя."
    )

## 13. Подбираем blending (если есть общая парная validation)

Blending — взвешенное усреднение вероятностей ViT и AST:

\[
P_{blend}=\alpha P_{image}+(1-\alpha)P_{audio}
\]

Подбираем `alpha` только на внутренней **общей размеченной** validation, где каждая строка относится к одному и тому же местоположению в обеих модальностях. Нельзя объединять предсказания просто потому, что у двух файлов одинаковый порядковый номер.

Если пар недостаточно, этот этап будет пропущен. Это нормально: тогда для итоговых папок `val` сохраняются отдельные предсказания каждой модели.

### Разбор перебора весов blending

`best_alpha = None` означает, что пока подходящий коэффициент не найден. `blend_results` будет хранить таблицу вариантов.

Если есть общая парная validation:
- `np.array_equal(image_eval_true, audio_eval_true)` проверяет, что метки в двух массивах идут в одном порядке и относятся к тем же объектам;
- `softmax(logits, axis=1)` переводит сырые logits каждой модели в вероятности по 13 классам;
- `np.linspace(0.0, 1.0, 21)` создаёт 21 вариант веса изображения от 0 до 1 с шагом 0.05. Вес аудио равен `1 - alpha`;
- `alpha * image_probs + (1-alpha) * audio_probs` объединяет вероятности двух моделей для одного и того же объекта;
- `.argmax(axis=1)` выбирает итоговый класс;
- `accuracy_score` и `f1_score` оценивают результат ансамбля по общим правильным меткам.

`pd.DataFrame(blend_rows)` собирает таблицу вариантов, `.sort_values(["macro_f1", "accuracy"], ascending=False)` сначала выбирает самый высокий Macro F1, а при равенстве — более высокую Accuracy. `.iloc[0]` берёт первую строку после сортировки; из неё извлекается лучший вес `alpha_image`.

Важно: подбор весов выполняется только на размеченной внутренней парной validation. Нельзя подбирать alpha по настоящей `val`, потому что там нет правильных ответов.

In [ ]:
best_alpha = None
blend_results = None

if BLENDING_EVAL_AVAILABLE:
    if not np.array_equal(image_eval_true, audio_eval_true):
        raise ValueError("Метки image/audio paired validation не совпадают по порядку.")

    image_probs_eval = softmax(image_eval_logits, axis=1)
    audio_probs_eval = softmax(audio_eval_logits, axis=1)
    blend_rows = []

    for alpha in np.linspace(0.0, 1.0, 21):
        blended_probs = alpha * image_probs_eval + (1.0 - alpha) * audio_probs_eval
        blended_pred = blended_probs.argmax(axis=1)
        blend_rows.append({
            "alpha_image": round(float(alpha), 2),
            "alpha_audio": round(float(1.0 - alpha), 2),
            "accuracy": accuracy_score(image_eval_true, blended_pred),
            "macro_f1": f1_score(
                image_eval_true,
                blended_pred,
                labels=np.arange(len(labels)),
                average="macro",
                zero_division=0,
            ),
        })

    blend_results = pd.DataFrame(blend_rows).sort_values(
        ["macro_f1", "accuracy"], ascending=False
    )
    display(blend_results)
    best_alpha = float(blend_results.iloc[0]["alpha_image"])
    print(f"Выбранный вес ViT: {best_alpha:.2f}; вес AST: {1-best_alpha:.2f}")
else:
    print("Подбор blending пропущен: нет общей парной validation.")

## 14. Получаем предсказания для ВСЕХ файлов в `image/val` и `audio/val`

Теперь используем `Trainer.predict()` не для одного примера, а для всего Dataset.

**Как работает пакетная обработка:** Trainer сам проходит по Dataset батчами. Размер батча при предсказании задаётся параметром `per_device_eval_batch_size` в `TrainingArguments`. Поэтому отдельный цикл по файлам писать не нужно.

Так как в настоящих `val` нет меток, `predict()` вернёт logits, но не Accuracy/F1.

### 14.1. Вспомогательная функция пакетного инференса

Функция:

1. переносит одну модель на рабочее устройство;
2. задаёт размер батча для инференса;
3. вызывает `trainer.predict(dataset)` для **всего** Dataset;
4. получает logits для каждого файла;
5. возвращает модель на CPU, чтобы освободить GPU для второй модальности.

### Разбор функции `predict_entire_dataset()`

Функция нужна, чтобы одинаково и безопасно получать предсказания для большого Dataset.

Аргументы:
- `trainer` — уже обученный Trainer (ViT или AST);
- `predict_dataset` — Dataset без меток или с метками, подготовленный через `with_transform`;
- `batch_size` — размер батча при инференсе;
- `description` — текст для понятного сообщения в выводе.

Строка `trainer.args.per_device_eval_batch_size = batch_size` задаёт размер батча для `predict()`. `trainer.model.to(trainer.args.device)` перемещает веса на устройство Trainer (обычно GPU, если оно доступно).

`trainer.predict(predict_dataset)` обходит весь Dataset пакетами, выполняет прямой проход и возвращает logits. Если модель вернула tuple, берётся первый элемент. Затем модель возвращается на CPU; `gc.collect()` очищает ненужные Python-объекты, `torch.cuda.empty_cache()` очищает свободный CUDA-кэш. `np.asarray(logits)` гарантирует, что наружу возвращается массив NumPy.

Функция возвращает матрицу формы `[число_файлов, число_классов]`. Она не вычисляет Accuracy, если у Dataset нет правильных меток.

In [ ]:
def predict_entire_dataset(trainer, predict_dataset, batch_size, description):
    trainer.args.per_device_eval_batch_size = batch_size
    trainer.model.to(trainer.args.device)

    print(f"Начинаем инференс: {description}")
    print("Количество объектов:", len(predict_dataset))
    print("Размер батча:", batch_size)

    prediction_output = trainer.predict(predict_dataset)
    logits = prediction_output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    trainer.model.to("cpu")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.asarray(logits)

### 14.2. Инференс на всех спутниковых изображениях

`image_val_ds` содержит все изображения из `image/val`, а `image_eval_transform` применяет к каждому изображению ту же детерминированную предобработку, что и при оценке модели.

Если в `image/val` 1000 файлов и `IMAGE_PREDICT_BATCH_SIZE=8`, Trainer обработает их примерно за 125 батчей. Последний батч может быть неполным.

### Разбор инференса по всему image/val

`IMAGE_PREDICT_BATCH_SIZE` берётся из размера evaluation-батча. При необходимости его можно уменьшить, если не хватает GPU-памяти.

`image_val.with_transform(image_eval_transform)` назначает детерминированную обработку всем изображениям из настоящей `image/val`. `predict_entire_dataset(...)` передаёт весь Dataset в `Trainer.predict()`, а Trainer сам делит его на батчи.

`softmax(image_val_logits, axis=1)` преобразует logits в вероятности. `argmax(axis=1)` выбирает ID с максимальной вероятностью. Список `id2label[...]` переводит ID в название класса. `.max(axis=1)` сохраняет максимальную оценку (confidence) для каждого файла.

Количество `image_val_pred_labels` должно совпадать с числом изображений в `image/val`. Здесь нет сравнения с правильными метками: это итоговый инференс.

In [ ]:
IMAGE_PREDICT_BATCH_SIZE = IMAGE_EVAL_BATCH_SIZE

image_val_ds = image_val.with_transform(image_eval_transform)
image_val_logits = predict_entire_dataset(
    image_trainer,
    image_val_ds,
    batch_size=IMAGE_PREDICT_BATCH_SIZE,
    description="все изображения image/val",
)

image_val_probs = softmax(image_val_logits, axis=1)
image_val_pred_ids = image_val_probs.argmax(axis=1)
image_val_pred_labels = [id2label[int(idx)] for idx in image_val_pred_ids]
image_val_confidence = image_val_probs.max(axis=1)

print("Получено предсказаний:", len(image_val_pred_labels))

### 14.3. Инференс на всех аудиозаписях

Точно так же обрабатываем `audio/val`. Функция `audio_transform` декодирует каждый OGG, приводит его к нужной частоте дискретизации и строит входные признаки AST.

Количество аудиофайлов может отличаться от количества изображений — это не мешает независимому инференсу двух моделей.

### Разбор инференса по всему audio/val

Логика такая же, как для изображений, но применяется `audio_transform`, которая декодирует OGG, приводит частоту дискретизации и готовит спектральные входные признаки AST.

`audio_val_decoded.with_transform(audio_transform)` создаёт Dataset-представление с обработкой при чтении. `predict_entire_dataset(...)` получает logits для всех аудиозаписей пакетами. Далее `softmax`, `argmax`, `id2label` и `.max(axis=1)` превращают logits в вероятности, названия классов и оценки уверенности.

Число аудиозаписей может быть не равно числу изображений. Это не проблема для независимого инференса: каждый файл получает собственное предсказание своей модели.

In [ ]:
AUDIO_PREDICT_BATCH_SIZE = AUDIO_EVAL_BATCH_SIZE

audio_val_ds = audio_val_decoded.with_transform(audio_transform)
audio_val_logits = predict_entire_dataset(
    audio_trainer,
    audio_val_ds,
    batch_size=AUDIO_PREDICT_BATCH_SIZE,
    description="все аудиозаписи audio/val",
)

audio_val_probs = softmax(audio_val_logits, axis=1)
audio_val_pred_ids = audio_val_probs.argmax(axis=1)
audio_val_pred_labels = [id2label[int(idx)] for idx in audio_val_pred_ids]
audio_val_confidence = audio_val_probs.max(axis=1)

print("Получено предсказаний:", len(audio_val_pred_labels))

## 15. Возвращаем имена исходных файлов и смотрим результаты

После `Trainer.predict()` у нас есть предсказания в том же порядке, в котором объекты находятся в Dataset. Получим пути к исходным файлам через `decode=False` — так не нужно повторно декодировать изображения и звук.

### Разбор функции извлечения имён файлов

**`get_unlabeled_file_records(dataset, column, feature_type, root_dir)`** возвращает метаданные файлов в том же порядке, в котором они находятся в Dataset.

- `dataset` — неразмеченный Dataset из `image/val` или `audio/val`;
- `column` — `"image"` или `"audio"`;
- `feature_type` — `HFImage` или `Audio`;
- `root_dir` — корневая папка, относительно которой нужно записать путь.

`cast_column(..., feature_type(decode=False))` просит Dataset вернуть путь к исходному файлу без декодирования его содержимого. Цикл по индексам сохраняет для каждого объекта его позицию `index`, путь `path`, относительное имя `filename` и `stem` без расширения.

`Path.resolve()` нормализует путь; `.relative_to(root_dir.resolve())` пытается получить путь относительно папки `val`; `.as_posix()` приводит разделители к `/`, что удобно для CSV и одинаково выглядит в разных ОС. Если путь не удаётся сделать относительным, `except` использует только имя файла.

Функции вызываются отдельно для image и audio. Затем `assert` проверяет, что число имён файлов в точности совпадает с числом предсказаний: это защищает от случайного сдвига строк при формировании CSV.

In [ ]:
def get_unlabeled_file_records(dataset, column, feature_type, root_dir):
    raw = dataset.cast_column(column, feature_type(decode=False))
    records = []

    for idx in range(len(raw)):
        item = raw[idx][column]
        path = item.get("path") if isinstance(item, dict) else None
        if not path:
            raise ValueError(f"Не удалось получить путь к файлу с индексом {idx}.")

        path_obj = Path(path)
        try:
            relative_name = path_obj.resolve().relative_to(root_dir.resolve()).as_posix()
        except Exception:
            relative_name = path_obj.name

        records.append({
            "index": idx,
            "path": path,
            "filename": relative_name,
            "stem": path_obj.stem,
        })

    return records


image_val_records = get_unlabeled_file_records(
    image_val, "image", HFImage, IMAGE_VAL_DIR
)
audio_val_records = get_unlabeled_file_records(
    audio_val, "audio", Audio, AUDIO_VAL_DIR
)

image_filenames = [row["filename"] for row in image_val_records]
audio_filenames = [row["filename"] for row in audio_val_records]

assert len(image_filenames) == len(image_val_pred_labels)
assert len(audio_filenames) == len(audio_val_pred_labels)

print("Image val files:", len(image_filenames))
print("Audio val files:", len(audio_filenames))

### 15.1. Формируем таблицы предсказаний

Для каждого файла сохраняем:

- `filename` — исходное имя файла;
- `predicted_label` — наиболее вероятный класс;
- `confidence` — максимальная softmax-оценка;
- `top_3` — три наиболее вероятных класса.

`confidence` — это оценка модели после Softmax, а не гарантированная вероятность правильного ответа.

### Разбор функции `make_prediction_frame()`

Функция превращает матрицу вероятностей в удобную таблицу.

Аргументы:
- `filenames` — имена исходных файлов в том же порядке, что и строки матрицы вероятностей;
- `probabilities` — массив формы `[число_файлов, число_классов]`, полученный после Softmax.

`probabilities.argmax(axis=1)` выбирает ID самого вероятного класса. `np.argsort(probabilities, axis=1)` возвращает индексы классов в порядке возрастания вероятности; `[:, -3:]` оставляет три самых высоких, а `[:, ::-1]` переворачивает их в порядке от наибольшей к меньшей.

В `pd.DataFrame` создаются колонки:
- `filename` — исходное имя/относительный путь;
- `predicted_label` — название класса с максимальной оценкой;
- `confidence` — максимальная Softmax-оценка;
- `top_3` — три наиболее вероятных класса, соединённые в одну строку через `" | "`.

`display(...head(10))` показывает первые десять строк для визуальной проверки. `.head(10)` не меняет весь DataFrame, а только возвращает его первые десять строк.

In [ ]:
def make_prediction_frame(filenames, probabilities):
    predicted_ids = probabilities.argmax(axis=1)
    top3_ids = np.argsort(probabilities, axis=1)[:, -3:][:, ::-1]

    return pd.DataFrame({
        "filename": filenames,
        "predicted_label": [id2label[int(idx)] for idx in predicted_ids],
        "confidence": probabilities.max(axis=1),
        "top_3": [
            " | ".join(id2label[int(idx)] for idx in row)
            for row in top3_ids
        ],
    })


image_predictions_df = make_prediction_frame(
    image_filenames, image_val_probs
)
audio_predictions_df = make_prediction_frame(
    audio_filenames, audio_val_probs
)

display(image_predictions_df.head(10))
display(audio_predictions_df.head(10))

## 16. Blending на настоящих `val` (если есть парные файлы)

В `val` нет правильных меток, поэтому здесь нельзя подбирать вес blending. Используем только `best_alpha`, подобранный ранее на размеченной парной внутренней validation.

Ищем уникальные совпадения по имени файла без расширения. Непарные файлы остаются с предсказанием своей модели. Одинаковое имя файла — лишь технический ключ; убедись, что в твоём датасете оно действительно означает один и тот же участок местности.

### Разбор поиска пар в настоящих val и blending

**`unique_stem_map(records)`** строит отображение `имя_без_расширения → индекс Dataset`, но сохраняет только те имена, которые встречаются ровно один раз. `Counter(...)` считает повторы; проверка `counts[row["stem"]] == 1` исключает неоднозначные имена.

`set(image_val_map) & set(audio_val_map)` находит ID, присутствующие и в image/val, и в audio/val. Сопоставление делается по stem, поэтому одинаковое имя JPG/OGG считается кандидатом в пару. Убедись, что имя файла действительно обозначает один и тот же участок; совпадение названия само по себе не доказывает семантическую связь.

Если `best_alpha` подобран ранее на размеченной общей validation и есть общие ID, списки `image_indices` и `audio_indices` извлекают соответствующие строки вероятностей из двух матриц. Затем применяется уже выбранная формула blending. Метки настоящей val неизвестны, поэтому качество здесь не вычисляется — сохраняется только прогноз ансамбля для парных файлов.

`blended_val_df.to_csv(...)` сохраняет результат. Непарные изображения и аудио не исчезают: их отдельные прогнозы всё равно находятся в `image_predictions_df` и `audio_predictions_df`.

In [ ]:
def unique_stem_map(records):
    counts = Counter(row["stem"] for row in records)
    return {
        row["stem"]: row["index"]
        for row in records
        if counts[row["stem"]] == 1
    }

image_val_map = unique_stem_map(image_val_records)
audio_val_map = unique_stem_map(audio_val_records)
common_val_stems = sorted(set(image_val_map) & set(audio_val_map))

print("Уникальных image val ID:", len(image_val_map))
print("Уникальных audio val ID:", len(audio_val_map))
print("Совпадающих ID:", len(common_val_stems))

blended_val_df = None
if best_alpha is not None and common_val_stems:
    image_indices = [image_val_map[stem] for stem in common_val_stems]
    audio_indices = [audio_val_map[stem] for stem in common_val_stems]

    blended_probs = (
        best_alpha * image_val_probs[image_indices]
        + (1.0 - best_alpha) * audio_val_probs[audio_indices]
    )
    blended_ids = blended_probs.argmax(axis=1)

    blended_val_df = pd.DataFrame({
        "sample_id": common_val_stems,
        "image_filename": [image_filenames[i] for i in image_indices],
        "audio_filename": [audio_filenames[i] for i in audio_indices],
        "predicted_label": [id2label[int(idx)] for idx in blended_ids],
        "confidence": blended_probs.max(axis=1),
    })

    display(blended_val_df.head(10))
    blended_val_df.to_csv(
        OUTPUT_DIR / "paired_blend_val_predictions.csv",
        index=False,
    )
else:
    print(
        "Blending на val пропущен: нет подобранного alpha или общих уникальных ID."
    )

## 17. Сохранение отдельных прогнозов и итогового CSV

Всегда сохраняем два отдельных файла:

- `image_val_predictions.csv` — все изображения из `image/val`;
- `audio_val_predictions.csv` — все аудиозаписи из `audio/val`.

Дополнительно собираем единый файл `M1_{lastname}_{firstname}_submission.csv`. В нём есть колонка `modality`, чтобы не смешивать одинаковые имена JPG и OGG, которые относятся к разным источникам данных.

> **Важное замечание:** точная схема submission зависит от требований организаторов. Если предоставлен шаблон CSV, названия столбцов и порядок строк нужно привести к нему. Сейчас используется понятная техническая схема `modality, filename, label`.

### Разбор сохранения CSV и объединения строк

`DataFrame.to_csv(path, index=False)` сохраняет таблицу в CSV. `index=False` не добавляет отдельную колонку с техническим индексом pandas.

Сначала отдельно сохраняются полные результаты image и audio, чтобы не потерять их раздельную принадлежность.

`pd.DataFrame({...})` формирует две таблицы одинаковой структуры. Значение `"image"` или `"audio"` записывается в колонку `modality` для каждой строки; `filename` хранит имя файла, `label` — предсказанный класс.

`pd.concat([...], ignore_index=True)` объединяет таблицы вертикально (строки одной под другой). `ignore_index=True` создаёт новый последовательный индекс после объединения.

Имя итогового файла формируется f-строкой с `LASTNAME` и `FIRSTNAME`. Затем `submission.to_csv(...)` сохраняет результат. **Перед отправкой обязательно проверь формат организаторов**: если в их шаблоне ожидаются другие колонки или только один источник данных, этот технический общий CSV нужно адаптировать.

In [ ]:
image_predictions_df.to_csv(
    OUTPUT_DIR / "image_val_predictions.csv",
    index=False,
)
audio_predictions_df.to_csv(
    OUTPUT_DIR / "audio_val_predictions.csv",
    index=False,
)

submission = pd.concat([
    pd.DataFrame({
        "modality": "image",
        "filename": image_filenames,
        "label": image_val_pred_labels,
    }),
    pd.DataFrame({
        "modality": "audio",
        "filename": audio_filenames,
        "label": audio_val_pred_labels,
    }),
], ignore_index=True)

submission_path = OUTPUT_DIR / f"M1_{LASTNAME}_{FIRSTNAME}_submission.csv"
submission.to_csv(submission_path, index=False)

print("Созданы файлы:")
print(OUTPUT_DIR / "image_val_predictions.csv")
print(OUTPUT_DIR / "audio_val_predictions.csv")
print(submission_path)

display(submission.head(10))

## 18. Итог эксперимента

В итоговой таблице сравниваются две модели по метрикам внутренней validation. Помни: если общей парной validation не было, результаты относятся к разным наборам объектов и не являются строгим прямым сравнением.

Сохранённые артефакты:

- `outputs/vit_location_model/` — лучшая версия ViT и её image processor;
- `outputs/ast_location_model/` — лучшая версия AST и её feature extractor;
- `outputs/image_val_predictions.csv` — предсказания для всех JPG;
- `outputs/audio_val_predictions.csv` — предсказания для всех OGG;
- `outputs/M1_..._submission.csv` — единая таблица предсказаний;
- `outputs/paired_blend_val_predictions.csv` — только если удалось подобрать blending и найти парные val-файлы.

### Разбор итоговой сводки

`pd.DataFrame([...])` создаёт компактную таблицу из метрик двух моделей. `image_metrics.get(...)` и `audio_metrics.get(...)` извлекают Accuracy и Macro F1; `.get()` безопасно возвращает `None`, если ключ не найден.

`display(summary)` показывает таблицу. Последующие `print()` выводят диагностическую информацию: доступна ли общая парная validation, какой коэффициент blending был выбран, сколько изображений и аудиозаписей обработано, и куда сохранены результаты.

Эта ячейка — финальная проверка: по ней удобно убедиться, что обе модели прошли обучение, предсказания рассчитаны для всех файлов и CSV записаны в ожидаемую папку.

In [ ]:
summary = pd.DataFrame([
    {
        "model": "ViT / image",
        "eval_accuracy": image_metrics.get("eval_accuracy"),
        "eval_macro_f1": image_metrics.get("eval_macro_f1"),
    },
    {
        "model": "AST / audio",
        "eval_accuracy": audio_metrics.get("eval_accuracy"),
        "eval_macro_f1": audio_metrics.get("eval_macro_f1"),
    },
])
display(summary)

print("Общая парная validation доступна:", BLENDING_EVAL_AVAILABLE)
print("Подобранный вес ViT для blending:", best_alpha)
print("Изображений в val:", len(image_filenames))
print("Аудиозаписей в val:", len(audio_filenames))
print("Папка результатов:", OUTPUT_DIR)

## 19. Что проверить перед сдачей

1. `DATA_ROOT` указывает на нужную папку, а внутри есть `image/train`, `image/val`, `audio/train`, `audio/val`.
2. В обоих `train` находятся одинаковые 13 классов. Но количество файлов по классам может различаться.
3. Настоящие `val` не имеют меток. Их используют только для предсказаний.
4. `Trainer.predict()` обрабатывает все элементы Dataset пакетами; размер батча задают `IMAGE_EVAL_BATCH_SIZE` и `AUDIO_EVAL_BATCH_SIZE`.
5. Блендинг разрешён только для действительно парных JPG/OGG. Не объединяй прогнозы только по позиции в списке.
6. Если включаешь `USE_CLASS_WEIGHTS=True`, повтори оба обучения и сравни новые метрики с базовыми.
7. Проверь формат CSV по инструкции организаторов. Не отправляй технический формат ноутбука, если шаблон соревнования требует другие столбцы.

### Полезная документация

- [Hugging Face Datasets — создание Dataset, ImageFolder и AudioFolder](https://huggingface.co/docs/datasets/create_dataset)
- [Hugging Face Datasets — Audio](https://huggingface.co/docs/datasets/audio_load)
- [Hugging Face Transformers — Image Classification](https://huggingface.co/docs/transformers/tasks/image_classification)
- [Hugging Face Transformers — Audio Classification](https://huggingface.co/docs/transformers/tasks/audio_classification)
- [Hugging Face Transformers — Trainer.predict](https://huggingface.co/docs/transformers/main_classes/trainer)
- [AST checkpoint](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593)

**Порядок работы:** сначала выполни установку, подключение Drive, настройки и загрузку данных. Проверь распределение классов и парные файлы. Затем запускай обучение ViT и AST. После обучения ноутбук сам выполнит пакетный инференс по всем файлам обеих папок `val` и сохранит CSV.